# Fock-PARFLM v2.1 — Proper Perplexity Evaluation (Debug Notebook)

## Purpose

This notebook performs an **independent, trustworthy perplexity (PPL) evaluation** of a
Fock-PARFLM v2.1 (d=384, L=16, M=32) checkpoint trained on OpenWebText.

It exists because the training loop’s in-line PPL estimate uses only 5 random batches
(≈40K tokens) from a 2M-token validation set, which can be noisy and unreliable.
This notebook instead:

1. **Scores the entire held-out validation set** (e.g. the full `openwebtext_val_2M.npy`),
   not a random subsample.
2. Uses a **strided sliding-window** approach so that (almost) every scored token sees
   a full `context`-length left context, rather than penalising tokens near window starts.
3. Reports PPL as an **aggregate** and as **mean ± std across chunks**, giving a
   confidence spread.

## Model Architecture

The checkpoint being evaluated is a **Fock-PARFLM v2.1** with:

| Component | Details |
|---|---|
| Base model | `FockMultiXiPARFLM` with `fock_version='v2'` |
| d / L / M | 384 / 16 / 32 |
| V_θ | `DepthConditionedMultiContextGaussianVTheta` (5 heads × 8 wells, shared bank + per-layer depth codes) |
| V_φ | `structural_competitive`, 4 heads, d_type=32, d_angle=16, top_k=16 |
| ξ channels | 5 (`'5long'`: α = [0.50, 0.75, 0.95, 0.99, 0.995]) |
| Reverse channel | Stable (QK-norm + soft-norm + pre-LN), per-layer gate |
| Embeddings | Untied W_out, output bias |
| Training context | 512 tokens (max_len=1024 but positions ≥512 never received gradients) |

## Important Gotchas

- **Eval at context=512, NOT 1024.** Positions ≥512 never received a gradient during
  training; evaluating at 1024 feeds untrained position embeddings.
- **`torch.no_grad()` compatibility.** The Fock force uses `autograd.grad(U, h)` internally.
  The eval script auto-detects this and falls back to grad-enabled forward with manual
  `detach()` if `no_grad()` fails.
- **CPU-only is RAM-constrained.** The grad-enabled fallback builds the full computation
  graph across 16 Verlet layers, consuming ~6–8 GB peak RAM per window. Use `BATCH=1`
  on free Colab (~12.7 GB). A full 2M-token eval takes ~2–3 hours at batch=1.

## Runtime

Designed for **Google Colab (CPU-only)**. No GPU required.

---
## 0. Configuration

Edit the paths below to match your Google Drive layout.

In [1]:
# ============================================================
# USER CONFIG — edit these paths to match your Drive layout
# ============================================================

# Run directory name on Google Drive (under MyDrive/)
RUN_DIR = 'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05'

# Checkpoint filename (inside RUN_DIR/checkpoints/)
#CKPT_NAME = 'fock_dcvt_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05_step77500_best.pt'
#CKPT_NAME = 'fock_dcvt_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05_step83500_best.pt'
CKPT_NAME = 'fock_dcvt_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05_step103500_best.pt'


# Validation data filename (inside RUN_DIR/data/)
VAL_NAME = 'openwebtext_val_2M.npy'

# Eval parameters
CONTEXT = 512       # MUST be 512 (native trained context)
STRIDE  = 256       # half-context overlap
BATCH   = 1         # MUST be 1 on free Colab CPU (12.7 GB RAM);
                    # the grad-enabled forward (needed by Fock autograd force)
                    # uses ~6-8 GB per window at d=384, L=16
N_CHUNKS = 20       # for mean +/- std computation

---
## 1. Mount Google Drive & Clone Repository

In [2]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Drive mounted.')
else:
    print('Not in Colab; assuming local paths.')

Mounted at /content/drive
Drive mounted.


In [3]:
# Clone the repo (needed for model source files)
REPO_DIR = Path('/content/semsimula-paper')

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone https://github.com/dimitarpg13/semsimula-paper.git /content/semsimula-paper
    else:
        print(f'{REPO_DIR} already exists, pulling latest...')
        !cd /content/semsimula-paper && git pull
else:
    # Local: adjust to your repo path
    REPO_DIR = Path(os.path.expanduser('~/git/ml/semsimula-paper'))

print(f'Repo dir: {REPO_DIR}')
assert REPO_DIR.exists(), f'Repo not found at {REPO_DIR}'

Cloning into '/content/semsimula-paper'...
remote: Enumerating objects: 6000, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 6000 (delta 6), reused 15 (delta 6), pack-reused 5985 (from 2)
Receiving objects: 100% (6000/6000), 938.66 MiB | 46.46 MiB/s, done.
Resolving deltas: 100% (3053/3053), done.
Updating files: 100% (2292/2292), done.
Filtering content: 100% (520/520), 78.32 MiB | 9.58 MiB/s, done.
Encountered 14 files that should have been pointers, but weren't:
	notebooks/conservative_arch/first_order_ablation/results/figures/headline_bar.png
	notebooks/conservative_arch/first_order_ablation/results/figures/trajectories.png
	notebooks/conservative_arch/first_order_ablation/results/splm1/seed0/splm_first_order_shakespeare_seed0_loss_curve.png
	notebooks/conservative_arch/first_order_ablation/results/splm1/seed1/splm_first_order_shakespeare_seed1_loss_curve.png
	notebooks/conservative_arch/first_order_ablation/results/

---
## 2. Set Up Python Path

The model files use `sys.path.insert` relative to `__file__` for their own
transitive imports (`model_parf` → `model_parf_sparse` → `model_parf_multixi`
→ `model_fock_parf_multixi`, plus `model_gaussian_vtheta` →
`model_structured_vtheta`). We just need the `parf/` directory on `sys.path`
so that the top-level imports resolve; everything else chains automatically.

In [4]:
PARF_DIR = str(REPO_DIR / 'notebooks' / 'conservative_arch' / 'parf')

# Also add the parent (conservative_arch/) for model_multixi, model_ln, etc.
CONSERVATIVE_DIR = str(REPO_DIR / 'notebooks' / 'conservative_arch')

for p in [PARF_DIR, CONSERVATIVE_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)
    # Also set PYTHONPATH for subprocess calls
    os.environ['PYTHONPATH'] = p + ':' + os.environ.get('PYTHONPATH', '')

print(f'PARF_DIR:         {PARF_DIR}')
print(f'CONSERVATIVE_DIR: {CONSERVATIVE_DIR}')
print(f'PYTHONPATH:       {os.environ["PYTHONPATH"][:200]}...')

PARF_DIR:         /content/semsimula-paper/notebooks/conservative_arch/parf
CONSERVATIVE_DIR: /content/semsimula-paper/notebooks/conservative_arch
PYTHONPATH:       /content/semsimula-paper/notebooks/conservative_arch:/content/semsimula-paper/notebooks/conservative_arch/parf:/env/python...


---
## 3. Verify Model Imports

Quick smoke test that all transitive model imports resolve before we spend
time loading the checkpoint.

In [5]:
try:
    from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
    print(f'FockMultiXiPARFLM:    {FockMultiXiPARFLM}')
    print(f'FockMultiXiPARFConfig: {FockMultiXiPARFConfig}')
except ImportError as e:
    print(f'IMPORT FAILED: {e}')
    print('Check that PARF_DIR is correct and the repo is cloned.')
    raise

try:
    from model_gaussian_vtheta import (
        DepthConditionedMultiContextGaussianVTheta,
        install_depth_routing,
    )
    print(f'DepthConditionedMultiContextGaussianVTheta: {DepthConditionedMultiContextGaussianVTheta}')
    print(f'install_depth_routing: {install_depth_routing}')
except ImportError as e:
    print(f'IMPORT FAILED: {e}')
    raise

print('\nAll model imports OK.')

FockMultiXiPARFLM:    <class 'model_fock_parf_multixi.FockMultiXiPARFLM'>
FockMultiXiPARFConfig: <class 'model_fock_parf_multixi.FockMultiXiPARFConfig'>
DepthConditionedMultiContextGaussianVTheta: <class 'model_gaussian_vtheta.DepthConditionedMultiContextGaussianVTheta'>
install_depth_routing: <function install_depth_routing at 0x7eebf09631a0>

All model imports OK.


---
## 4. Resolve Paths & Prepare Logfreq File

In [6]:
import numpy as np
import tempfile

if IN_COLAB:
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{RUN_DIR}')
else:
    # Local override: set to wherever the run dir lives
    GDRIVE_ROOT = Path(f'/path/to/local/{RUN_DIR}')

CKPT_PATH = GDRIVE_ROOT / 'checkpoints' / CKPT_NAME
VAL_PATH  = GDRIVE_ROOT / 'data' / VAL_NAME

print(f'Checkpoint: {CKPT_PATH}')
print(f'  exists: {CKPT_PATH.exists()}')
print(f'Val data:  {VAL_PATH}')
print(f'  exists: {VAL_PATH.exists()}')

assert CKPT_PATH.exists(), f'Checkpoint not found: {CKPT_PATH}'
assert VAL_PATH.exists(),  f'Val data not found: {VAL_PATH}'

# Create a logfreq file.
# The trained logfreq weights are INSIDE the checkpoint and will be overwritten
# by load_state_dict, so this dummy just needs to be the right shape (50257,)
# to let FockMultiXiPARFConfig.__init__ pass.
LOGFREQ_PATH = Path(tempfile.gettempdir()) / 'dummy_logfreq.npy'
_dummy_logfreq = np.full(50257, -np.log(1.0 / 50257), dtype=np.float32)
np.save(LOGFREQ_PATH, _dummy_logfreq)
print(f'Logfreq:   {LOGFREQ_PATH} (dummy, shape={_dummy_logfreq.shape})')

Checkpoint: /content/drive/MyDrive/semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05/checkpoints/fock_dcvt_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05_step103500_best.pt
  exists: True
Val data:  /content/drive/MyDrive/semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05/data/openwebtext_val_2M.npy
  exists: True
Logfreq:   /tmp/dummy_logfreq.npy (dummy, shape=(50257,))


---
## 5. Build Model & Load Checkpoint

This cell reproduces the exact model construction from
`colab_fock_depthcond_vtheta_openwebtext_ext2.ipynb`:

1. Build `FockMultiXiPARFLM` with `FockMultiXiPARFConfig(fock_version='v2', ...)`
2. Replace `model.V_theta` with `DepthConditionedMultiContextGaussianVTheta`
3. Install depth routing on `_fock_layer_step`
4. Patch `reverse_channel_scale` shape (per-layer `[16]` vs scalar `[1]`)
5. `load_state_dict`

A clean load (0 missing, 0 unexpected keys) confirms the config matches training.

In [7]:
import math
import torch
import torch.nn as nn

DEVICE = 'cpu'

# ── Resolved constants from the training notebook (XI_OVERRIDE='5long') ──
XI_ALPHA_INITS = [0.50, 0.75, 0.95, 0.99, 0.995]
XI_CHANNELS = len(XI_ALPHA_INITS)  # 5
D, L, M = 384, 16, 32

print(f'Building FockMultiXiPARFLM (d={D}, L={L}, M={M}, xi={XI_CHANNELS}ch)...')

config = FockMultiXiPARFConfig(
    vocab_size=50257, d=D, max_len=1024,
    L=L, v_hidden=1024, v_depth=3, dt=1.0,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_PATH),
    logfreq_init_alpha=0.1,
    init_gamma=1.0,
    fixed_gamma=0.30,
    causal_force=True,
    ln_after_step=True,
    xi_channels=XI_CHANNELS,
    xi_alpha_inits=XI_ALPHA_INITS,
    xi_learnable=True,
    xi_alpha_init_mode='explicit',
    v_phi_kind='structural_competitive',
    v_phi_d_type=32,
    v_phi_d_angle=16,
    v_phi_eps=0.1,
    v_phi_phi_hidden=128,
    v_phi_theta_hidden=128,
    v_phi_mlp_hidden=128,
    top_k=16,
    v_phi_n_heads=4,
    use_output_bias=True,
    tie_embeddings=False,
    score_head_hidden=32,
    gumbel_tau_init=1.0,
    gumbel_tau_min=0.3,
    gumbel_noise=True,
    use_gathered_v_phi=True,
    use_layer_checkpoint=True,
    ln_before_distance=True,
    per_layer_v_phi_scale=True,
    fock_version='v2',
    n_registers=M,
    register_salience_decay=0.5,
    register_salience_threshold=0.005,
    creation_gate_hidden=64,
    stack_discipline=True,
    d_k=64,
    tau_create_init=8.0,
    reverse_channel=True,
    reverse_channel_stable=True,
    reverse_channel_pre_ln=True,
    reverse_channel_soft_norm=True,
    reverse_channel_warmup_steps=4000,
    reverse_channel_per_layer=True,
    per_register_tau=True,
    per_register_keys=True,
    ortho_register_init=True,
    register_repulsion=True,
    register_repulsion_coeff=0.05,
    register_repulsion_kind='gram',
    # Legacy (leaky) register lifecycle: this notebook evaluates checkpoints
    # trained BEFORE the prefix-causal fix.  The model default is now True;
    # pin False so the forward semantics match the trained weights.
    prefix_causal_registers=False,
)

model = FockMultiXiPARFLM(config)
print(f'  base model created.')

# ── Replace V_theta with depth-conditioned Gaussian well bank ──
V_THETA_N_HEADS = XI_CHANNELS  # 5
V_THETA_WELLS_PER_HEAD = 8
_init_log_prec = -math.log(D)
_prec_max = 2.0 / D

model.V_theta = DepthConditionedMultiContextGaussianVTheta(
    d=D, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS,
    n_layers=L,
    w_scale=1.0,
    init_log_precision=_init_log_prec,
    precision_max=_prec_max,
    code_init_std=0.02,
).to(DEVICE)
install_depth_routing(model)
print(f'  V_theta replaced -> DepthConditionedMultiContextGaussian'
      f'({V_THETA_N_HEADS}h x {V_THETA_WELLS_PER_HEAD}w, L={L})')
print(f'  depth routing installed.')

# ── Load checkpoint ──
print(f'\nLoading checkpoint: {CKPT_PATH.name}...')
ckpt = torch.load(str(CKPT_PATH), map_location='cpu')
sd = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt

# Patch reverse_channel_scale shape if needed (per-layer [16] vs scalar [1])
if 'reverse_channel_scale' in sd:
    ckpt_shape = sd['reverse_channel_scale'].shape
    if hasattr(model, 'reverse_channel_scale') and model.reverse_channel_scale.shape != ckpt_shape:
        print(f'  patching reverse_channel_scale: '
              f'{model.reverse_channel_scale.shape} -> {ckpt_shape}')
        model.reverse_channel_scale = nn.Parameter(torch.zeros(ckpt_shape))

missing, unexpected = model.load_state_dict(sd, strict=False)
# reverse_warmup_step is a training-only counter, safe to ignore
unexpected = [k for k in unexpected if k != 'reverse_warmup_step']

if missing:
    print(f'\n  MISSING keys ({len(missing)}):')
    for k in missing[:15]:
        print(f'    {k}')
    if len(missing) > 15:
        print(f'    ... and {len(missing) - 15} more')

if unexpected:
    print(f'\n  UNEXPECTED keys ({len(unexpected)}):')
    for k in unexpected[:15]:
        print(f'    {k}')
    if len(unexpected) > 15:
        print(f'    ... and {len(unexpected) - 15} more')

if not missing and not unexpected:
    print('  load_state_dict: PERFECT MATCH (0 missing, 0 unexpected)')
else:
    print('\n  WARNING: key mismatch! Config does NOT match the checkpoint.')
    print('  PPL numbers from this model CANNOT be trusted.')

model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel ready on {DEVICE}: {n_params:,} parameters')

Building FockMultiXiPARFLM (d=384, L=16, M=32, xi=5ch)...
  base model created.
  V_theta replaced -> DepthConditionedMultiContextGaussian(5h x 8w, L=16)
  depth routing installed.

Loading checkpoint: fock_dcvt_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05_step103500_best.pt...
  load_state_dict: PERFECT MATCH (0 missing, 0 unexpected)

Model ready on cpu: 53,378,075 parameters


---
## 6. Inspect Checkpoint Keys (Optional Debug)

If the load above shows missing/unexpected keys, this cell helps diagnose
which parts of the config are wrong.

In [8]:
# Uncomment to inspect checkpoint contents

# print('Checkpoint top-level keys:', list(ckpt.keys()))
# print()
# print('State dict keys (first 40):')
# for i, k in enumerate(sorted(sd.keys())):
#     if i >= 40:
#         print(f'  ... ({len(sd)} total)')
#         break
#     print(f'  {k:60s}  {sd[k].shape}')

# Model vs checkpoint parameter shapes
# model_sd = {k: v.shape for k, v in model.state_dict().items()}
# ckpt_sd = {k: v.shape for k, v in sd.items()}
# for k in sorted(set(model_sd) | set(ckpt_sd)):
#     ms = model_sd.get(k)
#     cs = ckpt_sd.get(k)
#     flag = ''
#     if ms is None: flag = ' <-- UNEXPECTED (in ckpt, not in model)'
#     elif cs is None: flag = ' <-- MISSING (in model, not in ckpt)'
#     elif ms != cs: flag = f' <-- SHAPE MISMATCH (model={ms}, ckpt={cs})'
#     if flag:
#         print(f'  {k:60s}  model={ms}  ckpt={cs}{flag}')

---
## 7. Load Validation Data

In [9]:
val_tokens = np.load(str(VAL_PATH), mmap_mode='r')
val_tokens = np.asarray(val_tokens).reshape(-1)
print(f'Validation set: {val_tokens.shape[0]:,} tokens (dtype={val_tokens.dtype})')
print(f'  File: {VAL_PATH.name}')

Validation set: 2,000,000 tokens (dtype=uint16)
  File: openwebtext_val_2M.npy


---
## 8. Run Perplexity Evaluation

This uses the strided sliding-window evaluator from `eval_ppl_proper.py`.

- **context=512**: native trained context (positions ≥512 never saw a gradient)
- **stride=256**: each scored token has ≥256 tokens of left context
- **batch=1**: required on free Colab CPU (~12.7 GB); the grad-enabled forward
  (needed by Fock's internal `autograd.grad`) uses ~6–8 GB peak per window

In [ ]:
import gc
import torch.nn.functional as F

def _forward_logits(model, x, use_no_grad_flag):
    """Return logits (B, T, V). Auto-detects no_grad compatibility."""
    def _call():
        out = model(x)
        return out[0] if isinstance(out, (tuple, list)) else out

    if use_no_grad_flag[0]:
        try:
            with torch.no_grad():
                return _call().detach()
        except RuntimeError as e:
            print(f'[eval] torch.no_grad() failed ({e}); '
                  f'falling back to grad-enabled + detach.')
            use_no_grad_flag[0] = False
    return _call().detach()


@torch.no_grad()
def _nll_from_logits(logits, targets):
    """Per-position next-token NLL."""
    logp = F.log_softmax(logits.float(), dim=-1)
    return -logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)


def run_eval(model, tokens, context=512, stride=256, batch=1,
             device='cpu', n_chunks=20):
    assert 0 < stride <= context
    tokens = np.asarray(tokens).reshape(-1)
    N = tokens.shape[0]

    starts = list(range(0, N - context + 1, stride))
    if not starts:
        raise ValueError(f'val set too short ({N} tokens) for context={context}')

    use_no_grad_flag = [True]
    all_nll = []

    model.eval()
    for bstart in range(0, len(starts), batch):
        wstarts = starts[bstart:bstart + batch]
        xb = np.stack([tokens[s:s + context] for s in wstarts]).astype(np.int64)
        x = torch.from_numpy(xb).to(device)

        logits = _forward_logits(model, x, use_no_grad_flag)
        logits = logits[:, :-1, :]
        tgt = x[:, 1:]
        nll = _nll_from_logits(logits, tgt)

        for i, s in enumerate(wstarts):
            scored = nll[i] if s == 0 else nll[i, -stride:]
            all_nll.append(scored.detach().float().cpu())

        # Free the computation graph built by the grad-enabled forward.
        # Without this, dead graph nodes accumulate and OOM the session.
        del logits, nll, x, xb
        gc.collect()

        done = min(bstart + batch, len(starts))
        if done % max(1, len(starts) // 20) == 0 or done == len(starts):
            print(f'  windows {done}/{len(starts)}')

    all_nll = torch.cat(all_nll)
    n_tok = all_nll.numel()
    agg_loss = all_nll.mean().item()
    agg_ppl = math.exp(agg_loss)

    chunk_ppls = []
    csz = max(1, n_tok // n_chunks)
    for c in range(0, n_tok, csz):
        seg = all_nll[c:c + csz]
        if seg.numel() >= 32:
            chunk_ppls.append(math.exp(seg.mean().item()))
    chunk_ppls_t = torch.tensor(chunk_ppls)
    ppl_mean = chunk_ppls_t.mean().item()
    ppl_std = chunk_ppls_t.std(unbiased=True).item() if chunk_ppls_t.numel() > 1 else 0.0

    return {
        'tokens_scored': n_tok,
        'windows': len(starts),
        'context': context,
        'stride': stride,
        'agg_loss': agg_loss,
        'agg_ppl': agg_ppl,
        'ppl_mean_over_chunks': ppl_mean,
        'ppl_std_over_chunks': ppl_std,
        'n_chunks': int(chunk_ppls_t.numel()),
    }

print('Eval functions defined.')

In [ ]:
import time
import json
from datetime import datetime

print(f'Running PPL eval on {val_tokens.shape[0]:,} tokens...')
print(f'  context={CONTEXT}, stride={STRIDE}, batch={BATCH}')
print(f'  Expected windows: ~{(val_tokens.shape[0] - CONTEXT) // STRIDE + 1:,}')
print()

t0 = time.time()
result = run_eval(
    model, val_tokens,
    context=CONTEXT, stride=STRIDE, batch=BATCH,
    device=DEVICE, n_chunks=N_CHUNKS,
)
elapsed = time.time() - t0

print()
print('=' * 60)
print('RESULT: OpenWebText held-out (full)')
print('=' * 60)
print(f'  context / stride      : {result["context"]} / {result["stride"]}')
print(f'  windows               : {result["windows"]:,}')
print(f'  tokens scored         : {result["tokens_scored"]:,}')
print(f'  aggregate val_loss    : {result["agg_loss"]:.4f}')
print(f'  aggregate PPL         : {result["agg_ppl"]:.2f}   <-- headline number')
print(f'  PPL mean +/- std      : {result["ppl_mean_over_chunks"]:.2f} '
      f'+/- {result["ppl_std_over_chunks"]:.2f}  '
      f'(over {result["n_chunks"]} chunks)')
print(f'  wall time             : {elapsed:.1f}s')
print('=' * 60)

# ── Save result to Google Drive so it survives disconnects ──
if IN_COLAB:
    log_dir = Path(f'/content/drive/MyDrive/{RUN_DIR}/eval_logs')
    log_dir.mkdir(parents=True, exist_ok=True)

    log_entry = {
        'timestamp': datetime.now().isoformat(),
        'checkpoint': CKPT_NAME,
        'dataset': 'openwebtext_val_2M',
        'context': result['context'],
        'stride': result['stride'],
        'batch': BATCH,
        'tokens_scored': result['tokens_scored'],
        'windows': result['windows'],
        'agg_loss': round(result['agg_loss'], 6),
        'agg_ppl': round(result['agg_ppl'], 4),
        'ppl_mean_over_chunks': round(result['ppl_mean_over_chunks'], 4),
        'ppl_std_over_chunks': round(result['ppl_std_over_chunks'], 4),
        'n_chunks': result['n_chunks'],
        'wall_time_sec': round(elapsed, 1),
    }

    # Append to a JSONL log (one line per eval run)
    log_file = log_dir / 'eval_ppl_results.jsonl'
    with open(log_file, 'a') as f:
        f.write(json.dumps(log_entry) + '\n')
    print(f'\n✅ Result saved to Google Drive: {log_file}')

    # Also write a human-readable summary for quick inspection
    summary_file = log_dir / f'eval_{CKPT_NAME.replace(".pt", "")}_owt.txt'
    with open(summary_file, 'w') as f:
        f.write(f'Fock-PARFLM v2.1 Full-Set PPL Evaluation\n')
        f.write(f'========================================\n')
        f.write(f'Timestamp:  {log_entry["timestamp"]}\n')
        f.write(f'Checkpoint: {CKPT_NAME}\n')
        f.write(f'Dataset:    OpenWebText val (2M tokens)\n')
        f.write(f'Context:    {result["context"]}\n')
        f.write(f'Stride:     {result["stride"]}\n')
        f.write(f'Batch:      {BATCH}\n\n')
        f.write(f'Tokens scored:  {result["tokens_scored"]:,}\n')
        f.write(f'Windows:        {result["windows"]:,}\n\n')
        f.write(f'Aggregate loss: {result["agg_loss"]:.6f}\n')
        f.write(f'Aggregate PPL:  {result["agg_ppl"]:.4f}\n')
        f.write(f'PPL (chunks):   {result["ppl_mean_over_chunks"]:.4f} '
                f'+/- {result["ppl_std_over_chunks"]:.4f}\n')
        f.write(f'Wall time:      {elapsed:.1f}s\n')
    print(f'✅ Summary saved: {summary_file}')
else:
    print('\n(Not in Colab — skipping Drive save)')

---
## 9. Interpretation Guide

| PPL outcome | Interpretation |
|---|---|
| ≈16–17 (matches training log) | In-loop metric is trustworthy; the model genuinely achieves this PPL on the full val set |
| Noticeably higher (≈18–22) | In-loop estimate was biased low by small sample size or unlucky batches |
| Noticeably lower (≈14–15) | Possible train/val contamination in the 4B token pool |
| Very different (≥30 or NaN) | Config mismatch in `build_model`; check missing/unexpected keys above |

### Comparison baselines (OpenWebText, 512 context, GPT-2 BPE)

| Model | Params | PPL | Notes |
|---|---|---|---|
| GPT-2 small | 124M | ~29–32 | 1024 context, larger than Fock |
| Fock-PARFLM v2.1 d=384 | ~45M | **TBD** | This eval |

---
## 10. (Optional) WikiText-103 Cross-Corpus Check

If the OWT PPL looks suspiciously low, run the same eval on WikiText-103.
A model that memorised OWT will show a much larger gap than a GPT-2-class
model would.

In [ ]:
# Uncomment to run (requires: pip install datasets tiktoken)

# !pip install -q datasets tiktoken
# import tiktoken
# from datasets import load_dataset
#
# enc = tiktoken.get_encoding('gpt2')
# ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='test')
# text = '\n\n'.join(t for t in ds['text'] if t.strip())
# wt_tokens = np.asarray(enc.encode_ordinary(text), dtype=np.int32)
# print(f'WikiText-103 test: {wt_tokens.shape[0]:,} tokens')
#
# t0 = time.time()
# wt_result = run_eval(
#     model, wt_tokens,
#     context=CONTEXT, stride=STRIDE, batch=BATCH,
#     device=DEVICE, n_chunks=N_CHUNKS,
# )
# wt_elapsed = time.time() - t0
#
# print()
# print('=' * 60)
# print('RESULT: WikiText-103 test (cross-corpus check)')
# print('=' * 60)
# print(f'  aggregate PPL : {wt_result["agg_ppl"]:.2f}')
# print(f'  PPL mean +/- std : {wt_result["ppl_mean_over_chunks"]:.2f} '
#       f'+/- {wt_result["ppl_std_over_chunks"]:.2f}')
# print(f'  wall time : {wt_elapsed:.1f}s')
# print('=' * 60)
# print()
# print(f'OWT PPL: {result["agg_ppl"]:.2f}  vs  WT-103 PPL: {wt_result["agg_ppl"]:.2f}')
# print('If the gap is much larger than ~5-8 PPL, suspect OWT contamination.')
#
# # ── Save WikiText-103 result to Google Drive ──
# if IN_COLAB:
#     log_dir = Path(f'/content/drive/MyDrive/{RUN_DIR}/eval_logs')
#     log_dir.mkdir(parents=True, exist_ok=True)
#
#     wt_log_entry = {
#         'timestamp': datetime.now().isoformat(),
#         'checkpoint': CKPT_NAME,
#         'dataset': 'wikitext-103-test',
#         'context': wt_result['context'],
#         'stride': wt_result['stride'],
#         'batch': BATCH,
#         'tokens_scored': wt_result['tokens_scored'],
#         'windows': wt_result['windows'],
#         'agg_loss': round(wt_result['agg_loss'], 6),
#         'agg_ppl': round(wt_result['agg_ppl'], 4),
#         'ppl_mean_over_chunks': round(wt_result['ppl_mean_over_chunks'], 4),
#         'ppl_std_over_chunks': round(wt_result['ppl_std_over_chunks'], 4),
#         'n_chunks': wt_result['n_chunks'],
#         'wall_time_sec': round(wt_elapsed, 1),
#         'owt_agg_ppl': round(result['agg_ppl'], 4),
#         'owt_vs_wt103_gap': round(wt_result['agg_ppl'] - result['agg_ppl'], 4),
#     }
#
#     log_file = log_dir / 'eval_ppl_results.jsonl'
#     with open(log_file, 'a') as f:
#         f.write(json.dumps(wt_log_entry) + '\n')
#     print(f'\n✅ WikiText-103 result saved to Google Drive: {log_file}')
#
#     summary_file = log_dir / f'eval_{CKPT_NAME.replace(".pt", "")}_wt103.txt'
#     with open(summary_file, 'w') as f:
#         f.write(f'Fock-PARFLM v2.1 WikiText-103 Cross-Corpus Check\n')
#         f.write(f'================================================\n')
#         f.write(f'Timestamp:  {wt_log_entry["timestamp"]}\n')
#         f.write(f'Checkpoint: {CKPT_NAME}\n')
#         f.write(f'Dataset:    WikiText-103 test\n')
#         f.write(f'Context:    {wt_result["context"]}\n')
#         f.write(f'Stride:     {wt_result["stride"]}\n')
#         f.write(f'Batch:      {BATCH}\n\n')
#         f.write(f'Tokens scored:  {wt_result["tokens_scored"]:,}\n')
#         f.write(f'Windows:        {wt_result["windows"]:,}\n\n')
#         f.write(f'Aggregate loss: {wt_result["agg_loss"]:.6f}\n')
#         f.write(f'Aggregate PPL:  {wt_result["agg_ppl"]:.4f}\n')
#         f.write(f'PPL (chunks):   {wt_result["ppl_mean_over_chunks"]:.4f} '
#                 f'+/- {wt_result["ppl_std_over_chunks"]:.4f}\n')
#         f.write(f'Wall time:      {wt_elapsed:.1f}s\n\n')
#         f.write(f'--- Cross-corpus comparison ---\n')
#         f.write(f'OWT PPL:        {result["agg_ppl"]:.4f}\n')
#         f.write(f'WT-103 PPL:     {wt_result["agg_ppl"]:.4f}\n')
#         f.write(f'Gap (WT - OWT): {wt_result["agg_ppl"] - result["agg_ppl"]:.4f}\n')
#     print(f'✅ Summary saved: {summary_file}')

---

## Step 11 (CRITICAL): Trained-Scale Causal-Leak Probe + Honest PPL

The causal-leak audit measured the reverse-channel leak at **initialization scale** (~1e-5 logit shift) and explicitly required re-measuring it on the **trained** checkpoint (audit section 11, step 1). This matters because **every PPL number so far** (training `ntp`, in-loop eval, and the full-set sliding-window eval above) is computed via teacher-forced full-window forwards, where the Fock register summary is built from the *entire* 512-token window — including the scored token itself and its future. If training amplified the reverse-channel pathway (and the training log shows `reverse_channel_scale` is the dominant gradient group at nearly every step), all of those numbers are inflated coherently — and a WikiText-103 cross-check would **not** catch it, because the leak is architectural, not data contamination.

Two measurements (uses the `model` and `val_tokens` already loaded above):

1. **Part 1 — future-perturbation probe** (float64, a few minutes): swaps the future half of real val windows and measures the change in past-position logits and past-target NLL. The dNLL is the leak's predictive value in **nats/token**, directly comparable to the ~0.5–0.7 nat gap between the reported PPL (~9.5) and a plausible honest PPL (~16–20).
2. **Part 2 — honest PPL** (~30–90 min on CPU at k=1024; reduce `k` to 256 for a quick check): scores the same target tokens (a) mid-window with target+future inside the window (the standard protocol) and (b) from a window that ends *before* the target — leak-free by construction, with *more* left context. For a causal model (b) ≤ (a). If (b) ≫ (a), **PPL_B is the model's real perplexity.**

In [10]:
# ── Step 11: trained-scale leak probe + honest PPL ──────────────
import json
from datetime import datetime

DEBUG_DIR = str(REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup' / 'debug')
if DEBUG_DIR not in sys.path:
    sys.path.insert(0, DEBUG_DIR)

from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

# Part 1: future-perturbation probe at trained scale (float64, ~minutes)
probe_res = probe_trained_leak(model, val_tokens, device=DEVICE,
                               context=CONTEXT, n_pairs=4)

# Part 2: honest (leak-free) PPL vs standard protocol.
# k=1024 -> +/-0.08 nats; use k=256 for a fast first look (+/-0.16 nats).
HONEST_K = 1024
honest_res = honest_ppl_test(model, val_tokens, k=HONEST_K,
                             context=CONTEXT, batch=BATCH, device=DEVICE)

# ── Save results to Google Drive so they survive disconnects ──
if IN_COLAB:
    log_dir = Path(f'/content/drive/MyDrive/{RUN_DIR}/eval_logs')
    log_dir.mkdir(parents=True, exist_ok=True)
    entry = {
        'timestamp': datetime.now().isoformat(),
        'checkpoint': CKPT_NAME,
        'test': 'trained_leak_probe',
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }
    log_file = log_dir / 'leak_probe_results.jsonl'
    with open(log_file, 'a') as f:
        f.write(json.dumps(entry) + '\n')
    print(f'\n✅ Leak-probe results saved to Google Drive: {log_file}')

[probe] reverse gate tanh(scale) per layer: [0.0809, 0.0009, 0.0015, 0.0027, 0.0039, 0.0042, 0.0023, -0.014, -0.0087, -0.0082, 0.0128, 0.0113, 0.0097, 0.0137, 0.0216, 0.1656]
[probe] gate |mean| = 0.0226
[probe] 4 window pairs, context=512, t_p=256, dtype=float64
[eval] torch.no_grad() forward failed (element 0 of tensors does not require grad and does not have a grad_fn); falling back to grad-enabled forward with manual detach.
  [T0] determinism (same input twice): max|dlogit| = 0.000e+00
  pair 0: max|dlogit|(past) = 2.026e+01   mean dNLL(past targets) = +0.0271 nats
  pair 1: max|dlogit|(past) = 3.716e+01   mean dNLL(past targets) = -0.0042 nats
  pair 2: max|dlogit|(past) = 1.700e+01   mean dNLL(past targets) = +0.0273 nats
  pair 3: max|dlogit|(past) = 3.379e+01   mean dNLL(past targets) = -0.0017 nats
  [ctrl] gate zeroed: max|dlogit|(past) = 0.000e+00 (must be ~0)

[probe] SUMMARY
  trained-scale leak:  max|dlogit| at past positions = 3.716e+01   (init-scale reference ~1.1e-05)